In [ ]:
from langsmith import Client
import os
from dotenv import load_dotenv
load_dotenv()


client = Client()
print("LangSmith 연결 성공")
print(f"현재 프로젝트 : {os.getenv('LANGSMITH_PROJECT', 'default')}")

# 자동추적
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template(
    '''질문:{question}'''
)

chain= prompt | llm
result = chain.invoke({'question': "RAG란?"})
print(result)

# 커스텀 추척
from langsmith.run_helpers import traceable
@traceable(name = 'custom_rag_pipeline')  
def my_rag_duction(question:str)->str:
    # docs = retriever.inovoke(question)
    # answer = llm.invoke(format)
    # pass
    result = chain.invoke({'question':question})
    return result
    
result = my_rag_duction('langsmith란?')
print(result)

# langsmith 사이트에서 tracing project에서 확인가능함
# https://smith.langchain.com/


LangSmith 연결 성공
현재 프로젝트 : llm_rag_example
content='RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 자연어 생성(NLG) 기술을 결합한 방법론을 의미합니다. 이 접근 방식은 모델이 텍스트를 생성할 때 외부 데이터베이스나 문서에서 정보를 검색하여 이를 바탕으로 더 정확하고 관련성 높은 내용을 생성하는 방식입니다.\n\nRAG 모델의 주요 특징은 다음과 같습니다:\n\n1. **정보 검색**: 사용자가 입력한 질문이나 요청에 대해 관련 정보를 외부에서 검색합니다. 이 검색은 일반적으로 사전 훈련된 정보 검색 시스템을 사용합니다.\n   \n2. **텍스트 생성**: 검색된 정보를 바탕으로 자연어 생성 모델이 적절한 답변이나 문장을 생성합니다. 이 단계에서는 기계 학습 모델이 창의적으로 응답을 만들어냅니다.\n\n3. **효율성**: RAG는 관련된 정보를 실시간으로 검색하기 때문에, 사전 훈련된 모델만으로는 제공할 수 없는 최신 정보나 세부 사항을 포함할 수 있습니다.\n\n이러한 방식은 질문-답변 시스템, 대화형 AI, 콘텐츠 생성 등 다양한 자연어 처리 응용 분야에서 활용됩니다. RAG 모델은 최신 정보와 사용자 쿼리에 보다 정확하고 유용한 응답을 제공할 수 있는 능력을 가지고 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 284, 'prompt_tokens': 13, 'total_tokens': 297, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_toke

In [ ]:
# 이론 부분의 sample 코드에 대한 완전히 구현한 코드
import os
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from typing import Any,Dict,List
from dotenv import load_dotenv

# langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# langsmith
from langsmith import Client
from langsmith.run_helpers import traceable

# 환경설정
load_dotenv()
def check_environment():
    '''환경변수 확인'''
    missing_keys=[]
    if not os.getenv('OPENAI_API_KEY'):
        missing_keys.append('OPENAI_API_KEY')
    if not os.getenv('LANGCHAIN_API_KEY'):  #('LANGSMITH_API_KEY'):
        missing_keys.append('LANGCHAIN_API_KEY')  #('LANGSMITH_API_KEY')
    if missing_keys:s
        print('필요한 API key가 없습니다.')
        for key in missing_keys:
            print(f'--------{key}')
        return ValueError('필수 key 누락')
    
    # langsmith 추적 활성화
    # os.environ['LANGCHAIN_TRACING_V2']='true'
    # os.environ['LANGCHAIN_PROJECT']='llm_rag_example'
    print('환경설정 활성')

# langsmith 자동 출력
def auto_tracing():
    '''LangSmith를 기본 사용법'''
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ('system', '당신은 칠전할 ai agent 입니다. 사용자의 요구사항에 맞게 한글로 설명해주세요'),
        ('human', '간단히 설명해주세요: {topic}')
    ])
    chain = prompt | llm | StrOutputParser()
    topics = ['Python', 'AI']
    for topic in topics:
        response = chain.invoke({'topic':topic})
        print(f' {topic}: {response[:50]}...')
    print(f'자동출력 완료')


def traceable_decorator():
    '''커스텀 함수에 @traceable decorator를 사용해서 추척'''
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

    @traceable(name='custom_qa_function')
    def answer_question (question:str) -> str:
        '''질문에 답변하는 함수(langsmith에서 추적됨)'''
        prompt = f'질문에 간단히 답해주세요 : {question}'
        response = llm.invoke(prompt)
        return response.content
    
    @traceable(name='multi_step_analysis')
    def analyze_topic(topic:str) -> Dict[str,str]:
        '''여러 단계를 주제를 분석(중첩 추적)'''
        # 단계 1: 정의
        definition = answer_question(f'{topic}이란 무엇인가요?')
        # 단계 2: 장점
        advantage = answer_question(f'{topic}의 장점은?')

        return {
            'topic': topic,
            'definition': definition[:100],
            'advantage': advantage[:100]
        }
    
    print('\n@traceable 테스트')
    result = analyze_topic('LangChain')
    print(f"    주제: {result['topic']}")
    print(f"    정의: {result['definition']}")
    print(f"    강점: {result['advantage']}")
    print('\n@traceable 데코레이션 완료')


# 메탈데이터와 태그 추가
def metadata_tag():
    '''추적에 메타데이터와 태그를 추가해서 필터링/분석에 활용'''
    from langchain_core.runnables import RunnableConfig
    llm=ChatOpenAI(model='gpt-4o-mini', temperature=0)
    prompt=ChatPromptTemplate.from_template('{question}')
    chain = prompt | llm | StrOutputParser()

    # 메터데이터와 태그 설정
    config = RunnableConfig(
        meta={
            'user_id':'user_123',
            'session_id':'sess_456',
            'environment':'development',
            'version':'1.0.0'
        },
        tags=['example', 'qa', 'test']
    )
    print('\n메타데이터 / 태그 테스트')
    response = chain.invoke(
        {'question':'RAG란 무엇인가?'},
        config = config
    )

    print('\n메타데이터와 태그 추가 완료')


# LangSmith Client 직접 사용
def langsmith_client():
    '''LangSmith Client 를 직접 사용해서 데이터를 조회'''
    client = Client()
    print('\nLangSmith 프로젝트 목록 조회')
    try:
        projects = client.list_projects(limit=5)
        if projects:
            for project in projects :
                print(f' - {project.name}')
    except Exception as e:
        print(f'프로젝트 조회 중 오류 발생 : {e}')
    print('\n최근 실행기록')
    try:
        project_name = os.getenv('LANGCHAIN_PROJECT', 'default')
        runs = list(client.list_runs(
            project_name = project_name,
            limit=5
        ))
        if runs:
            for run in runs:
                status = 'success' if run.status == 'success' else 'faile'
                duration = run.end_time - run.start_time if run.start_time and run.end_time else 'N/A'
                cost = {run.total_cost}
                print(f' {status}{run.name} | {duration} | {cost}')
    except Exception as e:
        print(f'최근 실행기록 조회 중 오류 발생 : {e}')
    print('\n langSmith Client 사용완료')


def dataset_evaluation():
    '''langsmith에서 평가용 데이터셋을 생성하고 모델을 평가'''
    client = Client()
    #데이터셋이름 생성 (고유하게)
    dataset_name = f"qa_eval_dataset_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    import unicodedata
    # dataset_name = unicodedata.normalize('NFKD', dataset_name).encode('ascii', 'ignore').decode()
    # dataset_name = 'my_dataset_test'
    print(f'\n데이터셋 생성 : {dataset_name}')

    try:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description = 'QA 시스템 평가용 데이터셋'
        )

        # 평가용 예제
        examples = [
            {
                "inputs": {"question": "Python이란 무엇인가요?"},
                "outputs": {"answer": "Python은 프로그래밍 언어입니다."}
            },
            {
                "inputs": {"question": "1+1은?"},
                "outputs": {"answer": "2입니다."}
            },
            {
                "inputs": {"question": "AI란?"},
                "outputs": {"answer": "인공지능입니다."}
            }
        ]

        for ex in examples:
            client.create_example(
                inputs = ex['inputs'],
                ouputs = ex['outputs'],
                dataset_id = dataset.id
            )

        print(f' {len(examples)}개 예제 추가 완료')
        # 데이터셋 생성 내용 확인
        print('데이터셋 생성 내용 확인')
        saved_examples = client.list_examples(dataset_id=dataset.id)
        for i, ex in enumerate(saved_examples,1):
            question = ex.inputs.get('question', 'N/A')
            print(f'{i} {question}')

        # 테스트 로직

        # 정리(테스트 후 삭제)
        client.delete_dataset(dataset_id=dataset.id)
        print('데이터셋 삭제완료')

    except Exception as e:
        print(f'평가용 데이터셋 오류발생 {e}')





if __name__ == '__main__':
    check_environment()  # 환경체크
    auto_tracing()  # 자동추적
    traceable_decorator() # 커스텀 함수 추적
    metadata_tag() # 메타데이터와 태그 추가
    langsmith_client()  # 데이터 조회
    # dataset_evaluation() # 데이터셋 평가


    # # langsmith 개설
    # @traceable(name="create_traking_DJ")
    # def add(a,b):
    #     return a+b
    
    # add(1,2)



환경설정 활성


RuntimeError: Tracing using LangChainTracerV1 is no longer supported. Please set the LANGCHAIN_TRACING_V2 environment variable to enable tracing instead.

In [27]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)

{'messages': [HumanMessage(content='What is the weather in San Francisco?', additional_kwargs={}, response_metadata={}, id='72f3e191-8fb6-4d0c-b7a2-13d635984241'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 142, 'total_tokens': 230, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Citd3Uo8cgwUYH3ncsd1MnkgmNfSu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--666c91b5-27b5-41e3-bf0b-5aad7b11908b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_acoCYx0dxGzgXgPnKxJnBq5s', 'type': 'tool_call'}], usage_metadata={'input_tokens': 142, 'output_tokens': 88, '

In [4]:
# 이론부분의 sample 코드에 대한 완전히 구현한 코드
import os
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from typing import Any,Dict,List
from dotenv import load_dotenv

# langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# langsmith
from langsmith import Client
from langsmith.run_helpers import traceable

# 환경설정
load_dotenv()
def check_environment():
    '''환경변수 확인'''
    missing_keys = []
    if not os.getenv('OPENAI_API_KEY'):
        missing_keys.append('OPENAI_API_KEY')
    if not os.getenv('LANGCHAIN_API_KEY'):
        missing_keys.append('LANGCHAIN_API_KEY')
    if missing_keys:
        print('필요한 API키가 없습니다.')
        for key in missing_keys:
            print(f' --------- {key}')
        raise ValueError('필수 키 누락')
    
    # langsmith 추적 활성화
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_PROJECT'] = 'llm_rag_example'
    print('환경설정 완료!')

# langsmith  자동추적
def auto_tracing():
    '''langsmith 기본 사용법'''
    llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ('system','당신은 친절한 ai 에이전트입니다. 사용자의 요구사항에 맞게 한글로 설명해주세요'),
        ('human', '간단히 설명해주세요: {topic}')
    ])
    chain = prompt | llm | StrOutputParser()
    topics = ['Python','AI']
    for topic in topics:
        response = chain.invoke({'topic':topic})
        print(f'   {topic} : {response[:50]}...')
    print('자동추적 완료')

def traceable_decorator():
    '''커스텀함수에 @traceable 데코레이터를 사용해서 추적'''
    llm = ChatOpenAI(model = 'gpt-4o-mini',temperature=0)

    @traceable(name='custom_qa_function')
    def answer_question(question:str) -> str:
        '''질문에 답변하는 함수(langsmith에서 추적됨)'''
        prompt = f'질문에 간단히 답해주세요 : {question}'
        response = llm.invoke(prompt)
        return response.content
    @traceable(name="multi_step_analysis")
    def analyze_topic(topic:str)->Dict[str,str]:
        '''여러 단계로 주제를 분석(중첩 추적)'''
        # 단계 1: 정의
        definition = answer_question(f'{topic}이란 무엇인가요?')
        # 단계 2 : 장점
        advantage = answer_question(f'{topic}의 장점은?')

        return {
            'topic':topic,
            'definition' : definition[:100],
            'advantage' : advantage[:100],
        }
    print('\n@traceable 테스트')
    result = analyze_topic('LangChain')
    print(f"    주제 : {result['topic']}")
    print(f"    정의 : {result['definition']}")
    print(f"    강점 : {result['advantage']}")
    print('\n @traceable 데코레이터 완료!')

# 메탇이터 와 태그 추가    
def metadata_tag():
    '''추적에 메타데이터와 태그를 추가해서 필터링/분석에 활용'''
    from langchain_core.runnables import RunnableConfig
    llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)
    prompt = ChatPromptTemplate.from_template('{question}')
    chain = prompt | llm | StrOutputParser()
    # 메타데이터와 태그 설정
    config = RunnableConfig(
        metadata = {
            'user_id' : 'user_123',
            'session_id' : 'sees_456',
            'environment' : 'development',
            'version':"1.0.0"
        },
        tags = ['example','qa','test']
    )
    print('\n메타데이터 / 태그  테스트')
    response = chain.invoke(
        {'question':'RAG란 무엇인가요?'},
        config = config
    )
    print('\n메타데이터와 태그 추가 완료')


# langSmith Client 직접 사용
def langsmith_client():
    '''LangSmith client를 직접사용해서 데이터를 조회'''
    client = Client()
    print('\n프로젝트 목록 조회')
    try:
        projects = client.list_projects(limit=5)
        if projects:
            for project in projects:
                print(f'    - {project.name}')
    except Exception as e:
        print(f'프로젝트 조회중 오류 발생 : {e}')
    print('\n최근 실행기록')
    try:
        project_name = os.getenv('LANGCHAIN_PROJECT','default')
        runs = list(client.list_runs(
            project_name=project_name,
            limit=5
        ))
        if runs:
            for run in runs:
                status = 'success' if run.status == 'success' else 'faile'
                duration = run.end_time - run.start_time   if run.start_time and run.end_time else 'N/A'
                print(f'    {status} {run.name}  |  {duration}')
    except Exception as e:
        print(f'최근 실행기록 조회중 오류 발생 : {e}')
    print('\n langSmith Client 사용 완료')

def dataset_evaluation():
    '''langSmith에서 평가용 데이터셋을 생성하고 모델을 평가'''
    client = Client()
    # 데이터셋이름 생성(고유하게)
    dataset_name = f"qa_eval_dataset_{datetime.now().strftime('%Y%m%d_%H%M%S')}"    
    
    print(f'\n데이터셋 생성: {dataset_name}')

    try:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description='QA 시스템 평가용 데이터셋'
        )
        # 평가용 예제
        examples = [
            {
                "inputs": {"question": "Python이란 무엇인가요?"},
                "outputs": {"answer": "Python은 프로그래밍 언어입니다."}
            },
            {
                "inputs": {"question": "1+1은?"},
                "outputs": {"answer": "2입니다."}
            },
            {
                "inputs": {"question": "AI란?"},
                "outputs": {"answer": "인공지능입니다."}
            }
        ]
        for ex in examples:
            client.create_example(
                inputs=ex['inputs'],
                outputs=ex['outputs'],
                dataset_id=dataset.id
            )
        print(f'    {len(examples)}개 예제 추가 완료')
        # 데이터셋 생성 내용 확인
        print('데이터셋 생성 내용 확인')
        saved_examples = client.list_examples(dataset_id=dataset.id)
        for i, ex in enumerate(saved_examples,1):
            question = ex.inputs.get('question', 'N/A')
            print(f'  {i}  {question}')
        
        # 테스트 로직
        from langsmith.evaluation import evaluate
        client = Client()
        # 평가모델 정의
        llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)
        #평가 함수 실행
        def predict(inputs:str)->Dict[str,str]:
            q = inputs['question']
            result = llm.invoke(f'{q} 간단히 답해줘')
            return {'answer':result.content}
        def simple_correctness(run, example):
            """run.outputs 로 모델 답변을 가져오는 방식"""

            gold = example.outputs["answer"]
            pred = run.outputs["answer"]

            score = 1.0 if gold in pred else 0.0

            return {
                "key": "correctness",
                "score": score,
                "comment": f"gold={gold} | pred={pred}"
            }
        # 평가실행
        results = evaluate(
            predict,
            data=dataset_name,            
            evaluators=[simple_correctness]
        )

        print('\n평가 결과 요약')
        print(results)



        # 정리 (테스트 후 삭제)
        client.delete_dataset(dataset_id=dataset.id)
        print(' 데이터셋 삭제완료')
    except Exception as e:
        print(f' 평가용 데이터셋 오류발생 : {e}')



if __name__ =='__main__':
    check_environment()  #  환경체크
    auto_tracing() # 자동 추적
    traceable_decorator() # 커스텀 함수 추적
    metadata_tag() # 메타데이터 와 태그 추가
    langsmith_client()  # 데이터조회  client 사용
    dataset_evaluation()  # 평가용 데이터셋 생성 및 확인 그리고 삭제
    
    
    # @traceable(name="create_traking")
    # def add(a, b):
    #     return a + b

    # add(1, 2)

환경설정 완료!
   Python : Python은 고급 프로그래밍 언어로, 읽기 쉽고 배우기 쉬운 문법을 가지고 있습니다. 다...
   AI : AI(인공지능)는 컴퓨터나 기계가 인간처럼 학습하고 문제를 해결할 수 있도록 하는 기술입니...
자동추적 완료

@traceable 테스트
    주제 : LangChain
    정의 : LangChain은 자연어 처리(NLP) 애플리케이션을 구축하기 위한 프레임워크로, 다양한 언어 모델과 데이터 소스를 통합하여 복잡한 작업을 수행할 수 있도록 돕습니다. 주로 대화
    강점 : LangChain의 장점은 다음과 같습니다:

1. **모듈화**: 다양한 구성 요소를 조합하여 맞춤형 애플리케이션을 쉽게 만들 수 있습니다.
2. **확장성**: 여러 데이터 소

 @traceable 데코레이터 완료!

메타데이터 / 태그  테스트

메타데이터와 태그 추가 완료

프로젝트 목록 조회
    - llm_rag_example
    - default

최근 실행기록
    success ChatPromptTemplate  |  0:00:00
    faile ChatOpenAI  |  N/A
    faile RunnableSequence  |  N/A
    success custom_qa_function  |  0:00:03.541995
    success ChatOpenAI  |  0:00:03.541995

 langSmith Client 사용 완료

데이터셋 생성: qa_eval_dataset_20251204_140930
    3개 예제 추가 완료
데이터셋 생성 내용 확인
  1  AI란?
  2  1+1은?
  3  Python이란 무엇인가요?
View the evaluation results for experiment: 'artistic-candy-55' at:
https://smith.langchain.com/o/8e08528f-d22a-43e2-ae1c-bf871639e77c/datasets/2860

3it [00:07,  2.41s/it]



평가 결과 요약
<ExperimentResults artistic-candy-55>
 데이터셋 삭제완료
